### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)

#### This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:
    
##### . Grid Search/Random Search: Use grid search or random search to try different architectures.
##### . Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
##### . Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:

#####         o The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.

#####         o A common practice is to start with 1-2 hidden layers.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
# Load the dataset
data = pd.read_csv("Churn_Modelling.csv")

# Preprocess the data
# Drop unnecessary columns
data = data.drop(['RowNumber', 'CustomerId','Surname'], axis=1)

## Encode categorical variables
label_encoder_Gender = LabelEncoder()
data['Gender']= label_encoder_Gender.fit_transform(data['Gender']) 

# Onehot encode the 'Geography' column
geo_oh_encoder = OneHotEncoder(handle_unknown='ignore')
geo_encoder = geo_oh_encoder.fit_transform(data[['Geography']]).toarray()
geo_df = pd.DataFrame(geo_encoder, columns=geo_oh_encoder.get_feature_names_out(['Geography']))

# Combine the one-hot encoded columns with the original dataframe
data = pd.concat([data.drop('Geography', axis=1), geo_df], axis=1)

# Divide the dataset into independent and dependent features
x = data.drop('Exited', axis=1)
y=data['Exited'] 

# Split the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

# Scale the features

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Save the encoders and scalar for later use
with open('label_encoder_Gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_Gender, file)
with open('geo_oh_encoder.pkl', 'wb') as file:
    pickle.dump(geo_oh_encoder, file)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)




In [3]:
# Define a function to create the Keras classifier model and try different hyperparameters

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(x_train.shape[1],)))

    # Add additional hidden layers based on the 'layers' parameter
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model


In [4]:
# Create a keras classifier 

model= KerasClassifier(layers=1, neurons=32, build_fn=create_model, epochs=100, batch_size=10,verbose=0)



In [5]:
# Define Grid Search parameters

grid_params = {
    'neurons': [16,32,64,128],
    'layers': [1],
    'epochs': [50,100,150]

}

In [6]:
# Perform Grid Search
grid_search= GridSearchCV(estimator=model, param_grid=grid_params, cv=3, n_jobs=-1)
grid_results= grid_search.fit(x_train, y_train)

# Print the best parameters and best score

print("Best Parameters: %f using %s" % (grid_results.best_score_, grid_results.best_params_))


c:\Users\MSI\python\ANN_Project\fresh_env\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\MSI\python\ANN_Project\fresh_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Parameters: 0.858374 using {'epochs': 50, 'layers': 1, 'neurons': 128}
